In [2]:
import pandas as pd
from pygbif import species

In [3]:
df = pd.read_csv("..\output\observations.csv")
df.head()

,observationID,deploymentID,mediaID,eventID,eventStart,eventEnd,observationLevel,observationType,cameraSetupType,scientificName,...,bboxX,bboxY,bboxWidth,bboxHeight,classificationMethod,classifiedBy,classificationTimestamp,classificationProbability,observationTags,observationComments
0,671beca53724016fb5092fce,NaN,eb92d8f93b49494cd6e3ca8f7bb36e61,NaN,2024-05-15T16:23:36Z,2024-05-15T16:23:36Z,media,human,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,671bc0c62c50df857a79ebd4,2024-10-25T19:08:21Z,NaN,NaN,NaN
1,671becad3724016fb5092fd6,NaN,48fb2d4322923434be9d16e274ce27e8,NaN,2024-05-15T16:23:37Z,2024-05-15T16:23:37Z,media,human,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,671bc0c62c50df857a79ebd4,2024-10-25T19:08:29Z,NaN,NaN,NaN
2,671becaf34ad80e444d7dc77,NaN,1b74cb8cdca85b9a1a78364ddf15422c,NaN,2024-05-15T16:23:38Z,2024-05-15T16:23:38Z,media,human,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,671bc0c62c50df857a79ebd4,2024-10-25T19:08:31Z,NaN,NaN,NaN
3,671c155913678d7487bfa666,NaN,3960819f17101e4862d29d9200520be1,NaN,2024-03-13T14:47:18Z,2024-03-13T14:47:18Z,media,human,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,5e94888191fc50001775feb5,2024-10-25T22:02:01Z,NaN,NaN,NaN
4,671c155e9d1ec5c797309470,NaN,d29320f7846ad66b227ece842fde92da,NaN,2024-03-13T14:47:47Z,2024-03-13T14:47:47Z,media,human,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,5e94888191fc50001775feb5,2024-10-25T22:02:06Z,NaN,NaN,NaN


In [4]:
# 1. Clean your list: remove NaNs and non-string garbage
unique_names = [str(n).strip() for n in df["scientificName"].unique() if pd.notna(n)]


In [27]:

results = []

for name in unique_names:
    try:
        # Step 2: Search GBIF (strict=False helps with broad names like 'Reptilia')
        match = species.name_backbone(name=name, strict=False)
        usage_key = match.get('usageKey')
        
        if usage_key:
            # Step 3: Get common names
            v_data = species.name_usage(key=usage_key, data='vernacularNames')
            names_list = v_data.get('results', [])
            
            if names_list:
                # Priority: 1. English, 2. First available, 3. Skip
                common = next((i['vernacularName'] for i in names_list if i.get('language') == 'eng'), 
                              names_list[0]['vernacularName'])
                
                results.append({"scientificName": name, "commonName": common})
    except:
        continue

# Create final DataFrame
mapping_df = pd.DataFrame(results)

# Optional: Merge it back to your original data
# df = df.merge(mapping_df, on="scientificName", how="left")

print(mapping_df)

Empty DataFrame
Columns: []
Index: []


In [5]:
pd.DataFrame(unique_names, columns=["scientificName"]).to_csv("unique_scientific_names.csv", index=False)

In [6]:
df_names = pd.read_csv('../output/deployments.csv')

In [10]:
df_names['locationName'].to_csv("deployment_location_names.csv", index=False)
